<a href="https://colab.research.google.com/github/TheHighBrid/Fidelis/blob/main/notebooks/Fidelis_VOSR_MultiStep_Sweep_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fidelis: VOSR 1.4B Multi-Step Fidelity Sweep

Clean replacement for the failed Fidelis 2.1 source-fusion experiment.

Runs the official VOSR 1.4B multi-step model three times on the same source:
- Natural: CFG 0.50, weak condition 0.10
- Fidelity: CFG 1.25, weak condition 0.18
- Strict: CFG 1.75, weak condition 0.23

All runs use the same seed, 25 steps, wavelet colour alignment, deterministic VAE mode, and no post-processing.

**Colab compatibility:** Colab currently uses Python 3.13, while the official VOSR dependency pins target Python 3.11. Cell 1 therefore creates an isolated Python 3.11 environment for VOSR instead of modifying Colab's system Python.

Use a T4 GPU. Run Cells 1, 2, 3 and 4 in order.


In [ ]:
# @title 1. Prepare the official VOSR Python 3.11 environment
import os, shutil, subprocess, sys
from pathlib import Path

if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU runtime detected. Choose Runtime > Change runtime type > T4 GPU.")

ROOT = Path("/content/fidelis-multistep")
VOSR = ROOT / "VOSR"
FIDELIS = ROOT / "Fidelis"
VENV = ROOT / "venv311"
VOSR_COMMIT = "516f292b99cf23c76fdc33351e86dc4f97711fe8"
ROOT.mkdir(parents=True, exist_ok=True)

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
).strip().splitlines()[0]
print("GPU:", gpu)
print("Colab system Python:", sys.version.split()[0])

if not VOSR.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "https://github.com/cswry/VOSR.git", str(VOSR)],
        check=True,
    )
subprocess.run(["git", "-C", str(VOSR), "fetch", "--depth=1", "origin", VOSR_COMMIT], check=True)
subprocess.run(["git", "-C", str(VOSR), "checkout", "--detach", VOSR_COMMIT], check=True)

if not FIDELIS.exists():
    subprocess.run(
        ["git", "clone", "--depth=1", "https://github.com/TheHighBrid/Fidelis.git", str(FIDELIS)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(FIDELIS), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(FIDELIS), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(FIDELIS), "reset", "--hard", "origin/main"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"], check=True)
uv = shutil.which("uv")
if not uv:
    candidate = Path(sys.executable).parent / "uv"
    uv = str(candidate) if candidate.exists() else None
if not uv:
    raise RuntimeError("uv installed but its executable could not be located.")

if not (VENV / "bin" / "python").exists():
    print("Installing Python 3.11...")
    subprocess.run([uv, "python", "install", "3.11"], check=True)
    subprocess.run([uv, "venv", "--seed", "--python", "3.11", str(VENV)], check=True)

PY = VENV / "bin" / "python"
print("VOSR Python:", subprocess.check_output(
    [str(PY), "-c", "import sys; print(sys.version.split()[0])"], text=True
).strip())

marker = VENV / ".vosr_requirements_ok"
if not marker.exists():
    print("Installing official VOSR requirements inside Python 3.11...")
    subprocess.run([str(PY), "-m", "pip", "install", "--upgrade", "pip"], check=True)
    subprocess.run([str(PY), "-m", "pip", "install", "-r", str(VOSR / "requirements.txt")], check=True)
    marker.write_text(VOSR_COMMIT + "\n")

probe = r"""
import torch, numpy, diffusers, transformers
print("PyTorch:", torch.__version__, "| CUDA wheel:", torch.version.cuda)
print("NumPy:", numpy.__version__, "| diffusers:", diffusers.__version__, "| transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("VOSR Python cannot see the Colab GPU.")
p = torch.cuda.get_device_properties(0)
print("GPU:", p.name, "| VRAM:", round(p.total_memory / 1024**3, 1), "GB")
"""
subprocess.run([str(PY), "-c", probe], check=True)
print("Environment ready.")


In [ ]:
# @title 2. Upload the exact original test image
from google.colab import files
from pathlib import Path
import shutil

INPUT_DIR = ROOT / "input"
shutil.rmtree(INPUT_DIR, ignore_errors=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
allowed = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
saved = []
for original_name, data in uploaded.items():
    name = Path(original_name).name
    if Path(name).suffix.lower() not in allowed:
        continue
    target = INPUT_DIR / name
    target.write_bytes(data)
    saved.append(target)

if len(saved) != 1:
    raise RuntimeError(f"Upload exactly ONE original source image. Received {len(saved)}.")
SOURCE = saved[0]
print("Source:", SOURCE)


In [ ]:
# @title 3. Download only the required multi-step assets
import subprocess

CKPTS = VOSR / "preset" / "ckpts"
CKPTS.mkdir(parents=True, exist_ok=True)

download_code = r"""
from huggingface_hub import snapshot_download
from pathlib import Path
import sys

ckpts = Path(sys.argv[1])
snapshot_download(
    repo_id="CSWRY/VOSR",
    local_dir=str(ckpts),
    allow_patterns=[
        "VOSR_1.4B_ms/**",
        "Qwen-Image-vae-2d/**",
        "torch_cache/**",
    ],
)
"""
subprocess.run([str(PY), "-c", download_code, str(CKPTS)], check=True)

required = [
    CKPTS / "VOSR_1.4B_ms" / "args.json",
    CKPTS / "VOSR_1.4B_ms" / "checkpoints" / "ema_model.safetensors",
    CKPTS / "Qwen-Image-vae-2d" / "config.json",
    CKPTS / "Qwen-Image-vae-2d" / "diffusion_pytorch_model.safetensors",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise RuntimeError("Missing required files:\n" + "\n".join(missing))

for p in required:
    print("OK", p.relative_to(CKPTS), f"{p.stat().st_size / 1024**2:.1f} MB")


In [ ]:
# @title 4. Run all three profiles and download the ZIP
import os, shutil, subprocess, zipfile
from PIL import Image
from IPython.display import display
from google.colab import files

RESULTS = ROOT / "vosr-fidelity-results"
shutil.rmtree(RESULTS, ignore_errors=True)

sweep = FIDELIS / "experiments" / "vosr-fidelity-sweep.sh"
os.chmod(sweep, 0o755)

env = os.environ.copy()
env.update({
    "VOSR_DIR": str(VOSR),
    "VOSR_CKPT": "preset/ckpts/VOSR_1.4B_ms",
    "PYTHON_BIN": str(PY),
    "UPSCALE": "4",
    "INFER_STEPS": "25",
    "SEED": "42",
    "TILE_SIZE": "512",
})

print("Natural  : cfg=0.50 / weak=0.10")
print("Fidelity : cfg=1.25 / weak=0.18")
print("Strict   : cfg=1.75 / weak=0.23")
print("No source fusion, sharpening, face restoration, denoise or synthetic grain.")

subprocess.run([str(sweep), str(SOURCE), str(RESULTS)], env=env, check=True)

for profile in ("natural", "fidelity", "strict"):
    for p in sorted((RESULTS / profile).glob("*.png")):
        im = Image.open(p)
        print(f"\n{profile.upper()} | {im.width}x{im.height} | {p.stat().st_size / 1024**2:.2f} MB")
        preview = im.copy()
        preview.thumbnail((1200, 1200), Image.Resampling.LANCZOS)
        display(preview)

zip_path = ROOT / "vosr-fidelity-results.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in RESULTS.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(RESULTS.parent))

print("ZIP:", zip_path, f"{zip_path.stat().st_size / 1024**2:.1f} MB")
files.download(str(zip_path))
